In [1]:
import os
import math
import json
import warnings
import itertools

warnings.filterwarnings("ignore")

In [2]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [3]:
import numpy as np 
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.utils.preprocess import remove_price_dependency
from fintorch.utils.args import DefaultArgumentParser
from fintorch.utils.plot import draw_candlestick_plot, draw_labels, draw_predictions

In [4]:
string_args = [
    "--start-date", "2024-01-01",
    "--stop-date", "2024-06-01",
    "--time-frame", "60",
]

args = DefaultArgumentParser.parse(args=string_args)

In [5]:
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame).copy()

sdf

,open,high,low,close,volume,trade
timestamp,,,,,,
1705395660,43032.7,43040.0,42985.6,43003.7,201.772,3323
1705395720,43004.5,43016.0,42960.0,42967.3,534.556,4093
1705395780,42967.3,42976.5,42954.2,42957.8,117.148,1847
1705395840,42957.9,42965.7,42928.9,42942.2,245.105,3502
1705395900,42942.3,42953.0,42919.0,42942.6,134.988,1928
...,...,...,...,...,...,...
1711406100,70419.7,70438.9,70406.9,70438.8,79.550,1522
1711406160,70438.7,70524.7,70438.7,70510.3,134.209,2770
1711406220,70510.3,70510.4,70437.2,70464.4,78.321,1687


In [6]:
LENGTH = 5
LOOK_AHEAD = 11

df = sdf[-10000:]

# add label
df["sma"] = df.close.rolling(window=LOOK_AHEAD, center=True).mean()
df["fsma"] = df.sma.shift(periods=-LOOK_AHEAD // 2)
df["label"] = df.sma < df.fsma

# method 1 **
features = ["mean-close", "std-close"]
df["mean-close"] = df.close.rolling(window=LENGTH).mean()
df["std-close"] = df.close.rolling(window=LENGTH).std()
    

# # method 2
# features = ["tr", "roc", "vpt"]
# df["tr"] = df.high - df.low
# df["roc"] = df.close - df.open
# df["vpt"] = df.volume / df.trade
    
df = df.dropna() 
df

,open,high,low,close,volume,trade,sma,fsma,label,mean-close,std-close
timestamp,,,,,,,,,,,
1710806700,67804.5,67844.7,67781.8,67834.9,248.796,2333,67789.745455,67910.909091,True,67761.88,57.721720
1710806760,67834.9,67855.0,67800.0,67812.6,181.358,1940,67807.700000,67932.354545,True,67781.94,53.454448
1710806820,67812.7,67900.0,67805.2,67859.1,157.552,2636,67829.081818,67959.809091,True,67813.52,38.333236
1710806880,67859.1,67869.2,67844.7,67864.3,83.997,1164,67852.854545,67987.309091,True,67835.10,26.763688
1710806940,67864.2,67877.2,67826.3,67861.4,81.706,1510,67869.390909,68012.245455,True,67846.46,22.256302
...,...,...,...,...,...,...,...,...,...,...,...
1711405440,70409.7,70431.7,70384.7,70399.5,64.099,1302,70386.318182,70420.672727,True,70389.40,20.130574
1711405500,70399.6,70399.6,70354.1,70354.2,61.608,1465,70396.390909,70434.863636,True,70380.26,24.145662
1711405560,70354.2,70417.5,70350.6,70391.2,63.137,1429,70408.118182,70441.518182,True,70386.50,21.485926


In [7]:
SEQENCE_LENGTH = 32

x, y = [], []
for index in tqdm(range(SEQENCE_LENGTH, len(df))):
    fdf = df[features].iloc[index - SEQENCE_LENGTH: index]
    ndf = (fdf - fdf.min()) / (fdf.max() - fdf.min())
    # ndf = (ndf - ndf.mean()) / ndf.std()
    label = df.label.iloc[index]

    x.append(ndf.to_numpy())
    y.append(label)

x = np.array(x).reshape(-1, SEQENCE_LENGTH * len(features))
y = np.array(y)

print(x.shape, y.shape)

  0%|          | 0/9952 [00:00<?, ?it/s]

(9952, 64) (9952,)


In [16]:
TEST_LENGTH = LOOK_AHEAD
TRAIN_LENGTH = 10
WINDOW_LENGTH = TRAIN_LENGTH + TEST_LENGTH

print(TEST_LENGTH)

y_hat_list = []
correct, total = 0, 0
bar = tqdm(range(len(x)))
for start_index in bar:
    stop_index = start_index + WINDOW_LENGTH
    window_x, window_y = x[start_index: stop_index], y[start_index: stop_index]
    
    # split train and test set
    train_x, train_y = window_x[:TRAIN_LENGTH], window_y[:TRAIN_LENGTH]
    test_x, test_y = window_x[TRAIN_LENGTH:], window_y[TRAIN_LENGTH:]

    # evaluate model if there is enough samples
    if 0 < len(train_x) and 0 < len(test_x):
        # fit model
        model = LinearRegression(fit_intercept=False).fit(train_x, train_y)
        
        # evaluate model
        test_y_hat = model.predict(test_x)
        test_prediction = 0.5 < test_y_hat
        
        total += 1
        correct += (test_y[0] == test_prediction[0]).sum()
        accuracy= correct / total * 100
        bar.set_postfix_str("Accuracy: {:.2f}".format(accuracy))

        # print(test_y_hat.shape)
        y_hat_list.append(test_y_hat[0])

11


  0%|          | 0/9952 [00:00<?, ?it/s]

In [10]:
print(len(y_hat_list))
print(WINDOW_LENGTH)

9942
21


In [53]:
tdf = df[SEQENCE_LENGTH + TRAIN_LENGTH:].drop(columns=features)
tdf["prediction"] = 0.5 < np.array(y_hat_list)

tdf["fmax"] = tdf.close.rolling(21).max().shift(-LOOK_AHEAD + 1)
tdf["fmin"] = tdf.close.rolling(21).min().shift(-LOOK_AHEAD + 1)
tdf["ltp"] =  tdf.fmax / tdf.open - 1
tdf["stp"] =  tdf.open / tdf.fmin - 1

tdf["tr"] = tdf.high / tdf.low - 1
tdf["roc"] = tdf.close / tdf.open - 1
tdf["atr"] = tdf.tr.rolling(window=10).mean()
tdf["aroc"] = np.abs(tdf.roc).rolling(window=10).mean()


tdf["buy"] = tdf.prediction
tdf["sell"] = ~tdf.prediction
tdf["tp"] = tdf.aroc
tdf = tdf.dropna()

print((tdf.label == tdf.prediction).sum() / len(tdf))
print(tdf[tdf.prediction].ltp.mean())
print(tdf[~tdf.prediction].stp.mean())

print((tdf[tdf.buy].tp <= tdf[tdf.buy].ltp).sum() / tdf.buy.sum())
print((tdf[tdf.sell].tp <= tdf[tdf.sell].stp).sum() / tdf.sell.sum())

tdf

0.7165893972989317
0.0029571152104892824
0.002914410036652252
0.8744354110207768
0.8780487804878049


,open,high,low,close,volume,trade,sma,fsma,label,prediction,...,fmin,ltp,stp,tr,roc,atr,aroc,buy,sell,tp
timestamp,,,,,,,,,,,,,,,,,,,,,
1710809820,66971.9,67144.9,66947.8,67144.9,555.413,7594,67094.363636,67003.027273,False,False,...,66800.9,0.007349,0.002560,0.002944,0.002583,0.001906,0.001156,False,True,0.001156
1710809880,67144.8,67241.6,67117.4,67161.5,578.056,6991,67088.418182,66962.581818,False,False,...,66778.1,0.004228,0.005491,0.001850,0.000249,0.001954,0.001129,False,True,0.001129
1710809940,67161.3,67210.2,67109.2,67146.9,334.964,4183,67076.872727,66923.036364,False,False,...,66716.6,0.003431,0.006666,0.001505,-0.000214,0.002025,0.001095,False,True,0.001095
1710810000,67146.9,67155.7,67088.6,67108.1,200.552,3714,67061.045455,66895.554545,False,False,...,66711.9,0.002697,0.006521,0.001000,-0.000578,0.001900,0.000976,False,True,0.000976
1710810060,67108.1,67147.3,67036.7,67081.9,245.844,3428,67051.909091,66868.290909,False,False,...,66711.9,0.003277,0.005939,0.001650,-0.000390,0.001900,0.000932,False,True,0.000932
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1711404840,70352.2,70441.2,70350.0,70441.2,192.895,2637,70368.427273,70403.672727,True,True,...,70096.0,0.002435,0.003655,0.001296,0.001265,0.001213,0.000668,True,False,0.000668
1711404900,70441.2,70605.7,70441.2,70523.5,690.025,9220,70382.518182,70391.645455,True,True,...,70116.8,0.001168,0.004627,0.002335,0.001168,0.001380,0.000729,True,False,0.000729
1711404960,70523.4,70557.6,70392.9,70412.4,554.835,6690,70395.272727,70387.700000,False,True,...,70116.8,0.000001,0.005799,0.002340,-0.001574,0.001543,0.000861,True,False,0.000861


In [54]:
ENR = 0.5
TPR = 10
SLR = 10

#######################
###### backtest #######
#######################
bars_list, profits_list = [], []

order_index, order_price, entry_price, entry_index = None, None, None, None
for index in tqdm(range(len(tdf))):
    if entry_price is None:
        # reset order
        if order_index is not None and order_index + 10 < index:
            order_index = None
            order_price = None

        # set order
        if order_price is None and tdf.buy.iloc[index]:
            close_price = tdf.close.iloc[index]
            
            order_index = index
            order_price = round(close_price * (1 - tdf.aroc.iloc[index] * ENR), 2)

            # print(tdf.close.iloc[index])
            # print(order_price)
            # print()
            
        # check order status
        if order_price is not None and tdf.low.iloc[index] <= order_price:
            entry_index = index
            entry_price = order_price
            
            stop_loss = tdf.aroc.iloc[index] * SLR
            take_profit = tdf.aroc.iloc[index] * TPR
    
            stop_loss_price = round(entry_price * (1 - stop_loss - 0.0002 * 2), 2)
            take_profit_price = round(entry_price * (1 + take_profit + 0.0002 * 2), 2)
            
            # print(entry_price)
            # print(take_profit_price)
            # print(stop_loss_price)
            # print()
            
            order_index = None
            order_price = None
            
    else:
        profit = None
        if tdf.low.iloc[index] <= stop_loss_price:
            profit = -stop_loss
        elif take_profit_price <= tdf.high.iloc[index]:
            profit = take_profit

        if profit is not None:
            profits_list.append(profit)
            bars_list.append(index - entry_index)
            
            entry_index = None
            entry_price = None

  0%|          | 0/9922 [00:00<?, ?it/s]


65951.88
66826.08
65077.68

65158.84
66125.46
64192.22

64045.38
65290.29
62800.47

63367.48
64480.32
62254.64

64465.78
65191.71
63739.85

65267.73
66384.24
64151.22

63678.66
64212.7
63144.62

63890.91
64380.13
63401.69

61960.66
63392.68
60528.64

63121.73
64319.49
61923.97

62290.55
63321.57
61259.53

63239.26
63759.06
62719.46

61927.26
63524.88
60329.64

63632.73
64329.16
62936.3

64302.24
65262.45
63342.03

63222.76
64034.4
62411.12

64037.01
64622.22
63451.8

63590.15
64236.33
62943.97

62185.39
64256.09
60114.69

64488.98
66330.59
62647.37

66415.07
67441.49
65388.65

67387.27
68206.53
66568.01

68026.01
69096.22
66955.8

66661.7
67686.29
65637.11

67411.6
67780.55
67042.65

66721.53
67203.52
66239.54

67090.08
67537.09
66643.07

67506.11
68006.39
67005.83

67038.7
67493.33
66584.07

66120.45
66720.21
65520.69

66497.62
66799.76
66195.48

66823.3
67265.82
66380.78

65420.61
66339.93
64501.29

66239.91
66720.02
65759.8

65464.02
66049.57
64878.47

64866.71
65441.36
64292.06

6

In [51]:
LEVERAGE = 20

bars = np.array(bars_list)
profits = np.array(profits_list) * LEVERAGE

days = len(tdf) * TimeFrame.MIN1 // TimeFrame.DAY1
min_bars = bars.min()
max_bars = bars.max()
mean_bars = bars.mean()

trades = len(profits)
daily_trades = len(profits) // days

net_profit = profits.sum()
compound_profit = (1 + profits).prod() - 1
daily_net_profit = net_profit / days
daily_compound_profit = (compound_profit + 1) ** (1 / days) - 1

wining_ratio = (0 < profits).sum() / len(profits) * 100

# print statements
print("{:<32}{}".format("Days", days))
print()
print("{:<32}{}".format("Min bars", min_bars))
print("{:<32}{:}".format("Max bars", max_bars))
print("{:<32}{:.2f}".format("Mean bars", mean_bars))
print()
print("{:<32}{}".format("Trades", trades))
print("{:<32}{}".format("Daily trades", daily_trades))
print()
print("{:<32}{:.2f}".format("Net profit", net_profit))
print("{:<32}{:.2f}".format("Compound profit", compound_profit))
print("{:<32}{:.2f}".format("Daily net profit", daily_net_profit))
print("{:<32}{:.2f}".format("Daily compound profit", daily_compound_profit))
print()
print("{:<32}{:.2f}%".format("Wining ratio", wining_ratio))


Days                            6

Min bars                        6
Max bars                        636
Mean bars                       114.52

Trades                          79
Daily trades                    13

Net profit                      3.78
Compound profit                 4.43
Daily net profit                0.63
Daily compound profit           0.33

Wining ratio                    58.23%
